In [15]:
import pandas as pd
from pathlib import Path

repo_root = Path.cwd().parent.parent

In [16]:
#Dataset with the links of the youtube videos

domains = pd.read_csv(
    repo_root/"dataset"/"domains.csv.gz", 
    header=None,  
    names=['domain', 'count']  
)

print(domains.head(50))
print(f"Shape: {domains.shape}")

#  Filter domains that appear >= 1000 times 
domains_filtered_1000  = domains[domains['count'] >= 1000].copy()
print(f"Domains with >=1000 appearances: {len(domains_filtered_1000):,}")


                    domain     count
0              youtube.com  51357326
1                   bit.ly  29985037
2              twitter.com  27442504
3             facebook.com  26765796
4                   goo.gl  22548204
5            instagram.com  15901355
6                 youtu.be  12654279
7                  amzn.to  12161113
8                twitch.tv   5520111
9           soundcloud.com   3997308
10             patreon.com   3899918
11         plus.google.com   3563831
12              discord.gg   1881665
13        itunes.apple.com   1497118
14           pinterest.com   1474366
15             tinyurl.com   1399852
16              amazon.com   1246667
17         play.google.com   1197780
18     creativecommons.org    898587
19               paypal.me    841262
20             smarturl.it    805777
21                   po.st    759743
22      steamcommunity.com    706137
23           teespring.com    668093
24             go.magik.ly    647867
25                on.fb.me    608236
2

In [17]:
#Curlie dataset
curlie = pd.read_csv(
    repo_root/"dataset"/"curlie_filtered.csv.gz", 
)

print(curlie.head(50))
print(f"Shape: {curlie.shape}")


    Unnamed: 0                         url  uid  \
0            0      www.petpetgroup.com.hk    0   
1            1                  petsay.com    1   
2            2                 hkpearl.com    2   
3            3                      34c.cc    3   
4            4      www.petsfuneral.com.hk    4   
5            5            www.52993344.com    5   
6            6            www.52993344.com    5   
7            7      www.petparadise.com.tw    6   
8            8              www.koino1.com    7   
9            9         g66599.blogspot.com    8   
10          10      www.wondrouskennel.com    9   
11          11              www.hkherp.org   10   
12          12             uncyclopedia.tw   11   
13          13         www.gaoxiaohaha.com   12   
14          14             www.tktower.com   13   
15          15          www.outdoor.com.tw   14   
16          16                  rvcamp.biz   15   
17          17      www.paragliding-hk.com   16   
18          18            www.h

In [19]:
#Hm, some weird categories come up on top

print(curlie.columns.tolist())
print(f"Shape: {curlie.shape}")
print(curlie.iloc[400].to_frame().T) #sample row

#Okay, so there's different languages. We don't really want to filter out websites that have different languages, even if we focus on English ones in the YouNiverse dataset.
print("=== LANGUAGE DISTRIBUTION ===")
print(curlie['lang'].value_counts().head(10))

['Unnamed: 0', 'url', 'uid', 'label', 'lang', 'domain']
Shape: (933416, 6)
    Unnamed: 0             url   uid  \
400        400  www.shu.edu.tw  4854   

                                                 label lang      domain  
400  /en/Reference/Education/Colleges_and_Universit...   en  shu.edu.tw  
=== LANGUAGE DISTRIBUTION ===
lang
en    438171
de    153834
ja     55214
fr     45294
ru     35304
it     34875
nl     30547
pl     16186
tr     14404
da     14211
Name: count, dtype: int64


In [21]:
#testing different language categories

pl_data = curlie[curlie['lang'] == 'pl']
print(pl_data.head(5))

#Okay, so the languages have different categories depending on the language. Let's only focus on the english so far them and not worry too much about other languages.


      Unnamed: 0                 url    uid  \
1367        1367           lernu.net   5998   
1567        1567  www.gamedesire.com   6259   
1696        1696       www.avast.com   6399   
8467        8467          www.hp.com  20588   
8859        8859      www.debian.org  21084   

                                                  label lang          domain  
1367           /pl/Nauka_i_edukacja/J%C4%99zykoznawstwo   pl       lernu.net  
1567                                /pl/Gry/Internetowe   pl  gamedesire.com  
1696                /pl/Zakupy/Komputery/Oprogramowanie   pl       avast.com  
8467                                /pl/Komputery/Firmy   pl          hp.com  
8859  /pl/Komputery/Oprogramowanie/Systemy_operacyjn...   pl      debian.org  


In [22]:
curlie_en = curlie[curlie['lang'] == 'en'].copy()

curlie_en['full_path'] = curlie_en['label'].str.strip('/')
print(curlie_en.head(2))

print("="*10)

def extract_main_category_en(path):
    parts = path.strip('/').split('/')
    if len(parts) >= 2:
        return parts[1]  
    return path

curlie_en['main_category'] = curlie_en['full_path'].apply(extract_main_category_en)
print(curlie_en[['url', 'main_category']].head(5))


    Unnamed: 0                         url  uid  \
6            6            www.52993344.com    5   
24          24  www.hongkongdisneyland.com   24   

                                                label lang  \
6   /en/Recreation/Pets/Dogs/Breeds/NonSporting-Ut...   en   
24  /en/Recreation/Theme_Parks/Disney/Hong_Kong_Di...   en   

                    domain                                          full_path  
6             52993344.com  en/Recreation/Pets/Dogs/Breeds/NonSporting-Uti...  
24  hongkongdisneyland.com  en/Recreation/Theme_Parks/Disney/Hong_Kong_Dis...  
                           url main_category
6             www.52993344.com    Recreation
24  www.hongkongdisneyland.com    Recreation
86             www.the-afc.com        Sports
88              www.gcgcny.org       Society
92              www.kamkiu.com      Business


In [8]:
def analyze_domain_pattern(df, pattern, pattern_name, regex=True):
    matching_domains = df[df['url'].str.contains(pattern, case=False, na=False, regex=regex)]
    
    print(f"\n=== {pattern_name.upper()} ANALYSIS ===")
    print(f"{pattern_name} domains found: {len(matching_domains):,}")
    
    if len(matching_domains) > 0:
        if 'main_category' in matching_domains.columns:
            categories = matching_domains['main_category'].value_counts()
            for cat, count in categories.head(5).items():
                percentage = (count / len(matching_domains)) * 100
                print(f"{cat:15s}: {count:4,} ({percentage:5.1f}%)")
    
    
    return matching_domains

def analyze_youtuber_monetization_sites(df):
    
    monetization_patterns = {
        # E-commerce platforms
        '.shop domains': r'\.shop',
        'Shopify stores': r'shopify|myshopify',
        'Etsy shops': r'etsy\.com',
        'Amazon stores': r'amazon\.com|amzn\.to',
        
        # Creator platforms  
        'Patreon': r'patreon\.com',
        'Ko-fi': r'ko-fi\.com',
        'Buy Me a Coffee': r'buymeacoffee\.com',
        'Gumroad': r'gumroad\.com',
        
        # Merch platforms
        'Teespring/Spring': r'teespring\.com|spring\.com',
        'Redbubble': r'redbubble\.com',
        'Society6': r'society6\.com',
        'Spreadshirt': r'spreadshirt\.com',
        
        # Other ecommerce
        'Store extensions': r'\.(store|buy|sale|market|mall)',
        'Squarespace commerce': r'squarespace\.com',
        'WooCommerce': r'woocommerce\.com',
    }
    
    all_monetization = []
    
    for pattern_name, pattern in monetization_patterns.items():
        matching_domains = analyze_domain_pattern(df, pattern, pattern_name, regex=True)
        if len(matching_domains) > 0:
            all_monetization.append(matching_domains)
        print("-" * 50)
    
    if all_monetization:
        combined_monetization = pd.concat(all_monetization).drop_duplicates()
        
        print(f"\n=== COMBINED MONETIZATION ANALYSIS ===")
        print(f"Total unique monetization URLs: {len(combined_monetization):,}")
        
        if 'main_category' in combined_monetization.columns:
            print("\n=== CATEGORY BREAKDOWN ===")
            combined_cats = combined_monetization['main_category'].value_counts()
            for cat, count in combined_cats.head(10).items():
                percentage = (count / len(combined_monetization)) * 100
                print(f"{cat:20s}: {count:6,} ({percentage:5.1f}%)")
        
        return combined_monetization
    
    return pd.DataFrame()

monetization_sites = analyze_youtuber_monetization_sites(curlie_en)

print("\n" + "=" * 60)
print("AFFILIATE MARKETING ANALYSIS")
print("=" * 60)

affiliate_patterns = {
    'Amazon affiliates': r'amazon\.com.*tag=|amzn\.to',
    'ClickBank': r'clickbank',
    'Commission Junction': r'cj\.com|commission',
    'ShareASale': r'shareasale',
    'Affiliate networks': r'affiliate|referral|ref=',
}

for pattern_name, pattern in affiliate_patterns.items():
    analyze_domain_pattern(curlie_en, pattern, pattern_name, regex=True)
    print("-" * 30)


=== .SHOP DOMAINS ANALYSIS ===
.shop domains domains found: 118
Shopping       :   63 ( 53.4%)
Business       :   26 ( 22.0%)
Computers      :   14 ( 11.9%)
Society        :    5 (  4.2%)
Reference      :    3 (  2.5%)
--------------------------------------------------

=== SHOPIFY STORES ANALYSIS ===
Shopify stores domains found: 26
Shopping       :   22 ( 84.6%)
Business       :    2 (  7.7%)
Computers      :    1 (  3.8%)
Science        :    1 (  3.8%)
--------------------------------------------------

=== ETSY SHOPS ANALYSIS ===
Etsy shops domains found: 11
Shopping       :    6 ( 54.5%)
Arts           :    1 (  9.1%)
Business       :    1 (  9.1%)
Computers      :    1 (  9.1%)
Society        :    1 (  9.1%)
--------------------------------------------------

=== AMAZON STORES ANALYSIS ===
Amazon stores domains found: 4
Shopping       :    2 ( 50.0%)
Computers      :    1 ( 25.0%)
Society        :    1 ( 25.0%)
--------------------------------------------------

=== PATREON ANAL

C:\Users\veron\AppData\Local\Temp\ipykernel_26232\2189227247.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matching_domains = df[df['url'].str.contains(pattern, case=False, na=False, regex=regex)]



=== STORE EXTENSIONS ANALYSIS ===
Store extensions domains found: 412
Business       :  183 ( 44.4%)
Society        :   71 ( 17.2%)
Shopping       :   59 ( 14.3%)
Computers      :   31 (  7.5%)
Sports         :   14 (  3.4%)
--------------------------------------------------

=== SQUARESPACE COMMERCE ANALYSIS ===
Squarespace commerce domains found: 30
Sports         :    7 ( 23.3%)
Society        :    7 ( 23.3%)
Arts           :    5 ( 16.7%)
Reference      :    4 ( 13.3%)
Recreation     :    2 (  6.7%)
--------------------------------------------------

=== WOOCOMMERCE ANALYSIS ===
WooCommerce domains found: 0
--------------------------------------------------

=== COMBINED MONETIZATION ANALYSIS ===
Total unique monetization URLs: 667

=== CATEGORY BREAKDOWN ===
Business            :    255 ( 38.2%)
Shopping            :    159 ( 23.8%)
Society             :     88 ( 13.2%)
Computers           :     57 (  8.5%)
Sports              :     23 (  3.4%)
Reference           :     18 (  2.7

# ----------------------------------------
# ----------------------------------------

In [30]:
# Extract domain from Curlie URLs
def extract_domain(url):
    if pd.isna(url):
        return None
    domain = url.replace('http://', '').replace('https://', '').replace('www.', '')
    domain = domain.split('/')[0].split('?')[0].lower()
    return domain

curlie_en['domain'] = curlie_en['url'].apply(extract_domain)
domain_mapping = curlie_en[['domain', 'full_path']].drop_duplicates(subset=['domain']).reset_index(drop=True)

domains_matched = domains_filtered_1000.merge(
    domain_mapping, 
    on='domain', 
    how='left'
)

matched = domains_matched['full_path'].notna().sum()
total = len(domains_matched)

print(f"Curlie coverage: {matched:,}/{total:,} domains ({matched/total*100:.1f}%)")


Curlie coverage: 1,224/10,849 domains (11.3%)


In [35]:
print("\n=== TOP MATCHED DOMAINS WITH CATEGORIES ===")
matched_domains = domains_matched[domains_matched['full_path'].notna()].copy()
matched_domains['main_category'] = matched_domains['full_path'].apply(extract_main_category_en)

# Show top 20 matched domains with their categories
top_matched = matched_domains.nlargest(50, 'count')
print(top_matched[['domain', 'count', 'main_category', 'full_path']])

# Or in a cleaner format
print("\n=== TOP MATCHED DOMAINS (CLEAN FORMAT) ===")
for _, row in top_matched.iterrows():
    print(f"{row['domain']:25s}: {int(row['count']):8,} URLs -> {row['main_category']}")


=== TOP MATCHED DOMAINS WITH CATEGORIES ===
                     domain     count main_category  \
0               youtube.com  51357326     Computers   
2               twitter.com  27442504     Computers   
3              facebook.com  26765796     Computers   
10              patreon.com   3899918       Society   
16               amazon.com   1246667      Shopping   
17          play.google.com   1197780     Computers   
18      creativecommons.org    898587       Society   
26          incompetech.com    558463          Arts   
29               reddit.com    517217          News   
31   store.steampowered.com    491807      Shopping   
33               google.com    429736     Computers   
36               paypal.com    371354      Business   
41                 etsy.com    297104     Computers   
43             linkedin.com    287452      Business   
53            mediafire.com    193537     Computers   
54               flickr.com    189016     Computers   
57                vi

In [29]:

print("\n=== TOP UNMATCHED DOMAINS ===")
print(domains_matched[domains_matched['full_path'].isna()].nlargest(10, 'count')[['domain', 'count']])


=== TOP UNMATCHED DOMAINS ===
              domain     count
1             bit.ly  29985037
4             goo.gl  22548204
5      instagram.com  15901355
6           youtu.be  12654279
7            amzn.to  12161113
8          twitch.tv   5520111
9     soundcloud.com   3997308
11   plus.google.com   3563831
12        discord.gg   1881665
13  itunes.apple.com   1497118


# ------------------------------------------------------

In [34]:
print("=== ALL CURLIE MAIN CATEGORIES ===")
print(curlie_en['main_category'].value_counts())

=== ALL CURLIE MAIN CATEGORIES ===
main_category
Business          106434
Society            73031
Shopping           38929
Recreation         38294
Arts               37091
Computers          29052
Sports             28347
Science            24836
Reference          22195
Health             20686
Games               6784
Home                5029
Kids_and_Teens      4146
News                3317
Name: count, dtype: int64
